# ANÁLISIS DE VENTAS CON PANDAS
## Elias Buitrago Bolivar
###

En este cuaderno de jupyter se realiza un flujo de trabajo con pandas sobre una base de ventas de un supermercado. Se practica la selección de columnas, la localización con loc e iloc, el filtrado de filas, la creación de columnas derivadas, la agrupación con groupby y agg, y la unión de dos tablas con merge.

### Origen de los datos

Supermarket Sales, publicado en Kaggle por Aung Pyae.

Página del conjunto: https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales

Archivo de descarga directa (no requiere credenciales de Kaggle): https://raw.githubusercontent.com/plotly/datasets/master/supermarket_Sales.csv

Son 1000 registros de tres sucursales de Myanmar durante tres meses de 2019.

### Importar librerías np y pd

In [ ]:
import numpy as np
import pandas as pd

### Cargar datos

In [ ]:
url = 'https://raw.githubusercontent.com/plotly/datasets/master/supermarket_Sales.csv'
dataini = pd.read_csv(url)

In [ ]:
# Visualizar los primeros 5 registros incluyendo los encabezados
dataini.head()

In [ ]:
data = dataini.copy()
data.shape

### Conocer la estructura

In [ ]:
data.columns

In [ ]:
data.dtypes

In [ ]:
data.info()

In [ ]:
# Resumen estadístico de las columnas numéricas
data.describe()

### Normalizar los nombres de columna

In [ ]:
# Los encabezados traen espacios y mayúsculas, lo que complica cada línea posterior
data = data.rename(columns={'Tax 5%': 'Tax',
                            'Cost of goods sold': 'Cogs',
                            'Gross margin percentage': 'Gross margin pct',
                            'Customer stratification rating': 'Rating'})

In [ ]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
data.columns

### Seleccionar columnas

Antes de ejecutar las dos celdas siguientes: ¿por qué una devuelve una Series y la otra un DataFrame?

In [ ]:
# Una columna devuelve una Series
data['total'].head()

In [ ]:
# Dos corchetes devuelven un DataFrame
data[['product_line', 'total']].head()

In [ ]:
columnas = ['invoice_id', 'branch', 'city', 'product_line', 'quantity', 'unit_price', 'total']
ventas = data[columnas]
ventas.head()

In [ ]:
ventas.shape

### Localizar con loc

loc trabaja con etiquetas: nombres de columna y valores del índice. El rango incluye el extremo final.

In [ ]:
# Una celda: fila con índice 3, columna 'product_line'
data.loc[3, 'product_line']

In [ ]:
# Varias filas y varias columnas a la vez
data.loc[0:4, ['branch', 'product_line', 'total']]

In [ ]:
# Todas las filas de una lista de columnas
data.loc[:, ['branch', 'total']].head()

In [ ]:
# loc también acepta una condición en lugar de un rango
data.loc[data['branch'] == 'A', ['branch', 'city', 'total']].head()

### Localizar con iloc

iloc trabaja con posiciones enteras. El rango excluye el extremo final.

In [ ]:
# Fila 3, columna 5 por posición
data.iloc[3, 5]

In [ ]:
# Primeras 5 filas y primeras 4 columnas
data.iloc[0:5, 0:4]

In [ ]:
# La última fila
data.iloc[-1]

Antes de ejecutar la siguiente celda, escriba en un papel cuántas filas cree que devuelve cada una de las dos líneas.

In [ ]:
# La diferencia que sorprende: loc incluye el extremo final, iloc no
print(data.loc[0:2].shape[0])
print(data.iloc[0:2].shape[0])

### Filtrar filas

In [ ]:
# Una condición produce una máscara de True y False
mascara = data['branch'] == 'A'
mascara.head()

In [ ]:
sucursal_a = data[mascara]
sucursal_a.shape

In [ ]:
# Dos condiciones se combinan con & y cada una va entre paréntesis
mascara = (data['branch'] == 'A') & (data['total'] > 500)
data[mascara].shape

In [ ]:
# El operador | significa al menos una de las dos
mascara = (data['payment'] == 'Cash') | (data['payment'] == 'Ewallet')
data[mascara].shape

In [ ]:
# isin evita escribir varias comparaciones unidas con |
mascara = data['product_line'].isin(['Health and beauty', 'Sports and travel'])
data[mascara].shape

In [ ]:
# El operador ~ invierte la condición
mascara = ~(data['city'] == 'Yangon')
data[mascara]['city'].unique()

In [ ]:
# between incluye los dos extremos
mascara = data['total'].between(100, 200)
data[mascara].shape

In [ ]:
# Filtrado y selección de columnas en una sola instrucción con loc
seleccion = data.loc[(data['branch'] == 'C') & (data['gender'] == 'Female'),
                     ['city', 'product_line', 'quantity', 'total']]
seleccion.head()

In [ ]:
seleccion.shape

### Ordenar

In [ ]:
# Las diez ventas de mayor valor
data.sort_values('total', ascending=False).head(10)[['branch', 'product_line', 'total']]

In [ ]:
# nlargest hace lo mismo en una instrucción
data.nlargest(5, 'total')[['branch', 'product_line', 'total']]

### Crear columnas derivadas

In [ ]:
# La fecha llega como texto y hay que convertirla
data['date'] = pd.to_datetime(data['date'], format='%m/%d/%Y')
data['date'].dtype

In [ ]:
data['mes'] = data['date'].dt.month

In [ ]:
data['dia_semana'] = data['date'].dt.day_name()

In [ ]:
# La hora está en una columna aparte, también como texto
data['hora'] = pd.to_datetime(data['time'], format='%H:%M').dt.hour

In [ ]:
# El cálculo se aplica a la columna completa, sin escribir un ciclo
data['margen_pct'] = data['gross_income'] / data['total'] * 100
data[['total', 'gross_income', 'margen_pct']].head()

In [ ]:
# Antes de usar un indicador conviene mirar si varía. Aquí no varía:
# el margen es un 4.7619 % fijo en todas las ventas, así que no sirve para comparar
data['margen_pct'].describe()

In [ ]:
# Cuidado con el nombre que se le pone a una columna derivada.
# total ya incluye el impuesto del 5 %, así que esto no es el precio unitario:
# es el precio unitario con impuesto. El nombre debe decir lo que la columna contiene
data['precio_con_impuesto'] = data['total'] / data['quantity']
data[['unit_price', 'precio_con_impuesto']].head()

In [ ]:
# Comprobación: la razón entre los dos es constante e igual al impuesto
(data['precio_con_impuesto'] / data['unit_price']).round(4).unique()

In [ ]:
# np.where crea una columna condicional con dos salidas posibles
data['tamano_venta'] = np.where(data['total'] > 300, 'alta', 'normal')
data['tamano_venta'].value_counts()

In [ ]:
# pd.cut convierte una variable continua en bandas ordenadas
data['banda'] = pd.cut(data['total'],
                       bins=[0, 150, 400, 1100],
                       labels=['baja', 'media', 'alta'])
data['banda'].value_counts()

### Agrupar con groupby

In [ ]:
# Ingreso total por línea de producto
data.groupby('product_line')['total'].sum()

In [ ]:
# Número de facturas por sucursal
data.groupby('branch')['invoice_id'].count()

In [ ]:
# Promedio de la calificación por ciudad
data.groupby('city')['rating'].mean()

### Agregar con agg

In [ ]:
# Varias métricas a la vez, con nombres de columna explícitos
kpi_linea = (
    data
    .groupby('product_line')
    .agg(
        facturas=('invoice_id', 'size'),
        unidades=('quantity', 'sum'),
        ingreso=('total', 'sum'),
        rating=('rating', 'mean'),
    )
    .reset_index()
)
kpi_linea

In [ ]:
# El ticket promedio se calcula sobre los totales del grupo, no promediando promedios
kpi_linea['ticket_promedio'] = (kpi_linea['ingreso'] / kpi_linea['facturas']).round(2)
kpi_linea['rating'] = kpi_linea['rating'].round(2)
kpi_linea = kpi_linea.sort_values('ingreso', ascending=False)
kpi_linea

In [ ]:
# La línea de producto líder
kpi_linea.iloc[0]['product_line']

In [ ]:
# Agrupar por dos claves devuelve una fila por combinación existente
kpi_sucursal = (
    data
    .groupby(['branch', 'product_line'], as_index=False)
    .agg(ingreso=('total', 'sum'))
)
kpi_sucursal.head(10)

In [ ]:
kpi_sucursal.shape

agg devuelve una fila por grupo. Antes de ejecutar la celda siguiente: ¿cuántas filas cree que devuelve transform?

In [ ]:
# transform devuelve un valor por fila original, no uno por grupo
data['ingreso_sucursal'] = data.groupby('branch')['total'].transform('sum')
data['pct_de_sucursal'] = (data['total'] / data['ingreso_sucursal'] * 100).round(3)
data[['branch', 'total', 'ingreso_sucursal', 'pct_de_sucursal']].head()

In [ ]:
# pivot_table pasa el resultado a formato ancho
tabla = data.pivot_table(index='product_line',
                         columns='branch',
                         values='total',
                         aggfunc='sum',
                         fill_value=0)
tabla.round(2)

### Unir dos tablas con merge

La tabla de ventas no dice quién administra cada sucursal ni en qué zona está. Esa información vive en un catálogo aparte y se trae con merge.

In [ ]:
sucursales = pd.DataFrame({
    'branch': ['A', 'B', 'C'],
    'gerente': ['Laura Quintero', 'Andres Beltran', 'Paula Cifuentes'],
    'apertura': [2015, 2017, 2018],
    'zona': ['Centro', 'Norte', 'Sur'],
})
sucursales

In [ ]:
# how='left' conserva todas las ventas
# validate='many_to_one' detiene el proceso si la clave está repetida en el catálogo
data = data.merge(sucursales, on='branch', how='left', validate='many_to_one')
data.shape

In [ ]:
data[['branch', 'city', 'gerente', 'zona', 'total']].head()

El cruce anterior salió perfecto porque el catálogo tiene las tres sucursales. En el trabajo real los catálogos llegan incompletos, y conviene saber detectarlo antes de reportar cifras.

In [ ]:
# Un catálogo al que le falta la sucursal C
sucursales_parcial = sucursales[sucursales['branch'] != 'C']
sucursales_parcial

In [ ]:
# indicator agrega la columna _merge, que dice qué filas encontraron pareja
prueba = dataini.copy()
prueba.columns = prueba.columns.str.strip().str.lower().str.replace(' ', '_')
prueba = prueba.merge(sucursales_parcial, on='branch', how='left', indicator=True)
prueba['_merge'].value_counts()

In [ ]:
# Las 328 ventas de la sucursal C quedaron sin zona y sin gerente
prueba[prueba['_merge'] == 'left_only'][['branch', 'city', 'zona', 'gerente']].head()

In [ ]:
# Si no se revisa, un groupby por zona simplemente las ignora y el total no cuadra
print(prueba['total'].sum().round(2))
print(prueba.groupby('zona')['total'].sum().sum().round(2))

### Agrupar por un atributo que vino de la otra tabla

Este es el encadenamiento más frecuente del oficio: primero se cruza, después se resume por una columna que no estaba en la tabla original.

In [ ]:
kpi_zona = (
    data
    .groupby('zona', as_index=False)
    .agg(
        facturas=('invoice_id', 'size'),
        lineas=('product_line', 'nunique'),
        ingreso=('total', 'sum'),
        rating=('rating', 'mean'),
    )
)
kpi_zona['ticket_promedio'] = (kpi_zona['ingreso'] / kpi_zona['facturas']).round(2)
kpi_zona['rating'] = kpi_zona['rating'].round(2)
kpi_zona

In [ ]:
# Cruce de dos claves: zona y tipo de cliente
kpi_zona_cliente = (
    data
    .groupby(['zona', 'customer_type'], as_index=False)
    .agg(ingreso=('total', 'sum'))
)
kpi_zona_cliente

### Guardar el resultado

In [ ]:
data.to_csv('ventas_supermercado_procesado.csv', index=False)
kpi_linea.to_csv('kpi_por_linea_de_producto.csv', index=False)

In [ ]:
# Verificación final
print(data.shape)
print(data['total'].sum().round(2))
print(kpi_linea['ingreso'].sum().round(2))

### Y ahora la parte difícil

Este cuaderno trabajó sobre una tabla que llegó limpia: sin filas repetidas, sin celdas vacías y con una sola forma de escribir cada categoría. En el trabajo real eso casi nunca ocurre.

El siguiente cuaderno, 02 Preprocesamiento y ETL, toma una base de 119390 reservas de hotel con duplicados, faltantes, categorías sin definir y valores imposibles, y la deja en condiciones de responder las mismas preguntas que aquí se respondieron en una línea.